# SBOM-to-Audit — Stage 5.5 Historical Replay Checkpoint

This notebook independently verifies the CVE-2024-3400 / Operation MidnightEclipse historical replay from a clean GitHub checkout and an isolated Python environment.

It verifies both modes:

- public-evidence-only replay, with occurrence/publication separation and no fabricated local facts;
- synthetic reference-deployment EvidencePack replay.

It does not edit or push repository files.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with a Stage 5.5 tag after one is created.
print("Repository:", REPO_URL)
print("Reference:", REF)


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV_DIR = Path("/content/sbom_to_audit_stage55_venv")
for path in (WORKDIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Tested Git commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV_DIR)], check=True)
PYTHON = VENV_DIR / "bin" / "python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
import json
RELEASE_REPORT = Path("/content/stage55_colab_release_validation.json")
subprocess.run([str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)], check=True)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 51
print("PASS: canonical release gate")
print("Deterministic outputs:", len(release["deterministic_hashes"]))


In [ ]:
OUTPUT_ROOT = Path("/content/stage55_checkpoint_outputs")
ASSET_ROOT = Path("/content/stage55_checkpoint_assets")
for path in (OUTPUT_ROOT, ASSET_ROOT):
    if path.exists():
        shutil.rmtree(path)
subprocess.run([str(PYTHON), "scripts/run_historical_replay.py", "--output-root", str(OUTPUT_ROOT)], check=True)
subprocess.run([
    str(PYTHON), "-m", "sbom_to_audit.cli",
    "--scenario", "data/scenarios/historical_cve_2024_3400_reference.yaml",
    "--output-root", str(OUTPUT_ROOT),
], check=True)
subprocess.run([
    str(PYTHON), "paper_assets/scripts/build_stage55_assets.py",
    "--output-root", str(OUTPUT_ROOT), "--destination", str(ASSET_ROOT),
], check=True)


In [ ]:
import csv
import xml.etree.ElementTree as ET

public_bundle = json.loads((OUTPUT_ROOT / "historical_public/cve_2024_3400_public_bundle.json").read_text())
assert public_bundle["classification"] == "HISTORICAL_PUBLIC_REPLAY"
assert public_bundle["source_manifest"]["source_count"] == 7
assert len(public_bundle["timeline"]) == 8
assert public_bundle["evidence_boundaries"]["full_evidencepack_generated"] is False
assert public_bundle["evidence_boundaries"]["organisation_local_reachability"] == "unavailable"
assert public_bundle["provisional_source_ids"] == ["HIST-EPSS-001"]
assert public_bundle["manuscript_eligibility"] is False

with (OUTPUT_ROOT / "state_logs/historical_cve_2024_3400_reference.csv").open(newline="") as handle:
    states = list(csv.DictReader(handle))
assert [row["observed_state"] for row in states] == [
    "Monitor", "Report-Ready", "Report-Ready", "Report-Ready", "Report-Ready", "Report-Ready"
]
pack = json.loads((OUTPUT_ROOT / "evidence_packs/historical_cve_2024_3400_reference.json").read_text())
assert pack["vulnerability_intelligence"]["public_exploitation_observed"] is True
assert pack["local_evidence"]["malicious_exploitation_observed"] is False
assert pack["decision_state"]["authorized_state"] == "Report"
assert pack["asset_context"]["evidence_classification"] == "synthetic_reference_deployment"
ET.parse(ASSET_ROOT / "figures/cve_2024_3400_public_timeline.svg")
print("PASS: public-only evidence boundary")
print("PASS: occurrence/publication separation")
print("PASS: synthetic reference-deployment trajectory")
print("PASS: public exploitation is not local telemetry")
print("PASS: provisional EPSS blocks manuscript eligibility")


In [ ]:
import datetime as dt
import hashlib
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
    "evaluation_status": "CHECKPOINTED_PILOT_PROVISIONAL",
    "manuscript_eligibility": False,
}
ENV_PATH = Path("/content/stage55_colab_environment.json")
ENV_PATH.write_text(json.dumps(environment, indent=2) + "\n")
BUNDLE = Path("/content/stage55_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV_PATH, ENV_PATH.name)
    for root in (OUTPUT_ROOT, ASSET_ROOT):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("Tested Git commit:", COMMIT)
print("Colab evidence bundle SHA-256:", digest)
print("Download the ZIP from the Colab Files panel and preserve both values.")
